# V6 Task 2 — lightweight SVM ensemble

V6 adds balanced Linear SVM rankings to the strongest saved V3 semantic model. It uses word and character TF-IDF, takes roughly 20–60 seconds on this dataset, reads only `train.xlsx`, and does not generate a submission.

In [1]:
from pathlib import Path
import json
import pandas as pd
import v6_svm_factor_ensemble as v6

assert Path('train.xlsx').exists()
best_v3 = Path('outputs/v3_fast_semantic_c1.5')
if not (best_v3 / 'oof_predictions.npz').exists():
    best_v3 = Path('outputs/v3_fast_semantic')
assert (best_v3 / 'oof_predictions.npz').exists(), 'Run V3 evaluation first.'
print('Using V3 source:', best_v3)

Using V3 source: outputs/v3_fast_semantic_c1.5


In [2]:
cfg = v6.V6Config(v3_dir=str(best_v3))
metrics = v6.run_v6_oof('train.xlsx', cfg)
metrics

{
  "fold": 0,
  "rows": 330,
  "svm_c_0.03": 0.40007725823995416,
  "svm_c_0.1": 0.40229478275975517,
  "svm_c_0.3": 0.3919438955942838
}
{
  "fold": 1,
  "rows": 329,
  "svm_c_0.03": 0.3785450656730612,
  "svm_c_0.1": 0.3873681715336003,
  "svm_c_0.3": 0.39347662041689696
}
{
  "fold": 2,
  "rows": 327,
  "svm_c_0.03": 0.3651270379279035,
  "svm_c_0.1": 0.3774634828297505,
  "svm_c_0.3": 0.37515511873349716
}
{
  "fold": 3,
  "rows": 318,
  "svm_c_0.03": 0.3935497738099752,
  "svm_c_0.1": 0.4018431430703817,
  "svm_c_0.3": 0.4114601692333551
}
{
  "fold": 4,
  "rows": 331,
  "svm_c_0.03": 0.4015851922318329,
  "svm_c_0.1": 0.40606109757516634,
  "svm_c_0.3": 0.4066371874288954
}


{'rows': 1635,
 'users': 153,
 'v3_fixed_quota_macro_f1': 0.45397276946909826,
 'svm_fixed_quota_macro_f1': {'svm_c_0.03': 0.3998510651085785,
  'svm_c_0.1': 0.41165261756439425,
  'svm_c_0.3': 0.4066987662281538},
 'nested_blend_fixed_quota_macro_f1': 0.4430009069574092,
 'selected_blend_fixed_quota_macro_f1': 0.4621290429089626,
 'selected_blend_calibrated_macro_f1': 0.46112625119670075,
 'average_gold_labels': 2.9186544342507643,
 'average_calibrated_labels': 3.12782874617737,
 'fold_scores': [{'fold': 0,
   'rows': 330,
   'svm_c_0.03': 0.40007725823995416,
   'svm_c_0.1': 0.40229478275975517,
   'svm_c_0.3': 0.3919438955942838},
  {'fold': 1,
   'rows': 329,
   'svm_c_0.03': 0.3785450656730612,
   'svm_c_0.1': 0.3873681715336003,
   'svm_c_0.3': 0.39347662041689696},
  {'fold': 2,
   'rows': 327,
   'svm_c_0.03': 0.3651270379279035,
   'svm_c_0.1': 0.3774634828297505,
   'svm_c_0.3': 0.37515511873349716},
  {'fold': 3,
   'rows': 318,
   'svm_c_0.03': 0.3935497738099752,
   'svm_c

## Read the result

Compare `selected_blend_calibrated_macro_f1` with the V3 calibrated score. Also inspect the nested score: if it is lower, the apparent improvement may partly come from per-label selection and should be treated cautiously.

In [3]:
result = json.loads(Path('outputs/v6_svm_factor_ensemble/oof_metrics.json').read_text())
print('V3 fixed:', round(result['v3_fixed_quota_macro_f1'], 4))
print('V6 nested blend:', round(result['nested_blend_fixed_quota_macro_f1'], 4))
print('V6 selected fixed:', round(result['selected_blend_fixed_quota_macro_f1'], 4))
print('V6 selected calibrated:', round(result['selected_blend_calibrated_macro_f1'], 4))

V3 fixed: 0.454
V6 nested blend: 0.443
V6 selected fixed: 0.4621
V6 selected calibrated: 0.4611


In [4]:
per_label = pd.read_csv('outputs/v6_svm_factor_ensemble/oof_per_label.csv')
per_label.sort_values('f1')[['factor', 'support', 'svm_source', 'svm_weight', 'precision', 'recall', 'f1']]

,factor,support,svm_source,svm_weight,precision,recall,f1
18,sexual orientation related issues,8,svm_c_0.03,0.0,0.142857,0.125000,0.133333
16,cognitive deficits,33,svm_c_0.1,0.2,0.181818,0.181818,0.181818
2,substance use,33,svm_c_0.03,0.0,0.190476,0.242424,0.213333
13,exposure to others' suicide,14,svm_c_0.03,0.0,0.214286,0.214286,0.214286
23,meaning in life,45,svm_c_0.3,0.5,0.200000,0.244444,0.220000
6,poor school performance,16,svm_c_0.1,0.2,0.363636,0.250000,0.296296
1,physical health/characteristic,78,svm_c_0.03,0.0,0.274809,0.461538,0.344498
7,low socio-economic status,54,svm_c_0.03,0.0,0.338235,0.425926,0.377049
22,sense of responsibility,58,svm_c_0.03,0.0,0.379310,0.379310,0.379310
19,social support,112,svm_c_0.03,0.0,0.373333,0.500000,0.427481
